In [1]:
import cv2
import numpy as np
import tensorflow as tf

CLASS_COLORS = {
    1: [255,   0,   0],   # ship -> red
    2: [128, 128, 128],   # sargassum -> gray
    3: [255, 255, 255],   # oil -> white
}

# Load trained model
model = tf.keras.models.load_model("./segmentation_unet.h5", compile=False)

# Open video
cap = cv2.VideoCapture("../video/ManchaSat.mp4")

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

def colorize_mask(mask):
    # Create a 3-channel RGB canvas
    mask_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    # Ships = class 1 -> red
    mask_rgb[mask == 1] = (0, 0, 255)

    # Oil = class 6 -> white
    mask_rgb[mask == 3] = (255, 255, 255)

    # Sargassum = class 7 -> orange (BGR)
    mask_rgb[mask == 2] = (0, 165, 255)

    return mask_rgb

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    input_tensor = preprocess_frame(frame)
    pred = model.predict(input_tensor, verbose=0)

    mask = decode_mask(pred, frame.shape)
    mask_colored = colorize_mask(mask)

    # Show only the mask (not the original frame)
    cv2.imshow("Segmentation Mask", mask_colored)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


2025-08-27 13:57:08.506935: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-27 13:57:08.528835: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-27 13:57:08.528858: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-27 13:57:08.528862: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-27 13:57:08.532648: I tensorflow/core/platform/cpu_feature_g